In [ ]:
import os
from pathlib import Path
import pandas as pd
from itables import show
import requests
import time


In [ ]:
df_fliway = pd.read_excel('data/raw/fliway_source_to_geocode.xlsx')
df_fliway

In [ ]:
df_fliway.columns = df_fliway.iloc[0, :]
df_fliway = df_fliway.iloc[1:]
df_fliway 



In [ ]:
IATA_list = df_fliway['Charge Location IATA Code'].value_counts().index.tolist()
IATA_list

In [ ]:
to_geocode = df_fliway[['Suburb','Post Code (text)', 'Delivery Depot', 'Charge Zone', 'Distance (Kms)']]
to_geocode = to_geocode.drop_duplicates()
to_geocode = to_geocode.rename(columns={'Distance (Kms)': 'Distance (Google)'})
to_geocode

In [ ]:
def delete(row):
    if row:
        words = row.split()
        for word in words:
            if word in IATA_list:
                return None
        return row     

to_geocode['Suburb'] = to_geocode['Suburb'].apply(delete)
to_geocode.dropna(inplace=True)
to_geocode


In [ ]:
to_geocode.loc[to_geocode['Suburb'] == 'Arthur s Pass National Park', 'Suburb'] = "Arthur's Pass National Park"
to_geocode[to_geocode['Suburb'].str.contains("Arthur")]

In [ ]:
API_KEY = os.getenv("MAPBOX_ACCESS_TOKEN")
if not API_KEY:
    raise RuntimeError("Set MAPBOX_ACCESS_TOKEN before running Mapbox geocoding cells.")


def geocode_structured(suburb, postcode):
    url = (
        f"https://api.mapbox.com/search/geocode/v6/forward"
        f"?locality={requests.utils.quote(suburb)}"
        f"&postcode={requests.utils.quote(str(postcode))}"
        f"&country={requests.utils.quote('New Zealand')}"
        f"&access_token={API_KEY}"
    )

    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('features'):
                coords = data['features'][0]['geometry']['coordinates']
                feature_name = data['features'][0]['properties']['name']
                feature_type = data['features'][0]['properties']['feature_type']

                print(f'processed {suburb} {postcode}')
                return coords[0], coords[1], feature_name, feature_type  # lon, lat
    except Exception as e:
        print(f"Error for {lon}, {lat}: {e}")
    return None, None, None, None



In [ ]:
print(geocode_structured('Abbey Caves', '0110'))

In [ ]:
to_geocode[['Longitude (mapbox)', 'Latitude (mapbox)', 'Feature_Name (mapbox)', 'Feature_Type (mapbox)']] = to_geocode.apply(
    lambda row: pd.Series(geocode_structured(row['Suburb'], row['Post Code (text)'])),
    axis=1 
)

In [ ]:
def validate_geocodes(row):
    feature_name = row['Feature_Name (mapbox)']
    feature_type = row['Feature_Type (mapbox)']
    suburb = row['Suburb']
    postcode = row['Post Code (text)']
    
    if feature_type == 'locality':
        if feature_name in suburb:
            return True 
        else:
            return False
    elif feature_type == 'postcode':
        if feature_name in str(postcode):
            return True
        else:
            return False


within the mapbox response data structure, the feature type and feature name that was geocoded is also included. 



In [ ]:

API_KEY = os.getenv("MAPBOX_ACCESS_TOKEN")
if not API_KEY:
    raise RuntimeError("Set MAPBOX_ACCESS_TOKEN before running Mapbox geocoding cells.")

def reverse_geocode(lon, lat):
    url =(
        f"https://api.mapbox.com/search/geocode/v6/reverse"
        f"?longitude={requests.utils.quote(str(lon))}"
        f"&latitude={requests.utils.quote(str(lat))}"
        f"&access_token={API_KEY}"
    )

    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('features'):
                for feature in data['features']:
                    feature_type = feature['properties'].get('feature_type')
                    if feature_type in ('locality', 'postcode'):
                        feature_name = feature['properties'].get('name')
                        print(f'processed {lon}, {lat}')
                        return feature_name
    except Exception as e:
        print(f"Error for {lon}, {lat}: {e}")
    return None



In [ ]:
to_geocode['Validate'] = to_geocode.apply(validate_geocodes, axis=1)
to_geocode


In [ ]:
len(to_geocode[to_geocode['Validate'] == False])

In [ ]:
print(f"{round(len(to_geocode[to_geocode['Validate'] == False])/len(to_geocode)* 100, 2)}%")

In [ ]:
Path('data/output').mkdir(parents=True, exist_ok=True)
to_geocode.to_csv('data/output/geocoded_structured.csv', index=False)